In [ ]:
import pandas as pd
import requests
import re
import json
import time

In [ ]:
API_URL = "https://api.perplexity.ai/chat/completions"  # Ajustar según documentación actual
API_KEY = "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"  # Reemplazar con tu clave de API real


def query_perplexity(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "sonar-pro",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1000,
        "temperature": 0.7
    }
    response = requests.post(API_URL, json=payload, headers=headers)
    if response.status_code == 200:
        data = response.json()
        return data["choices"][0]["message"]["content"]
    else:
        raise Exception(f"Error {response.status_code}: {response.text}")


def augment_dataframe(df):
    # Problem Description	Python Code	Source	Description Error	Error Label
    df_subset = df.copy()

    problem_description=[]
    python_code=[]
    source=[]
    description_error=[]
    error_label=[]

    problems_aug = []
    descriptions_aug = []
    k=1
    for _, row in df_subset.iterrows():
        print("row ",k)
        k+=1
        problem_text = row[0]
        succefull=False
        while not succefull:
          try:
            prompt_problem = f"Give me 5 paraphrases of '{problem_text}' in comma-separated format. like this paraphrase1,paraphrase2,paraphrase3, paraphrase4,paraphrase5"
            print("Execute "+prompt_problem)
            result_problem = query_perplexity(prompt_problem)
            results_problem=re.split(r',\s*\n*',result_problem)
            print("result problem\n\n",results_problem,"\ntam:",len(results_problem))
            if(len(results_problem)==5):
              succefull=True
              print("Succefully")
          except:
            print("Error")
            time.sleep(2)

        time.sleep(2)

        succefull=False
        while not succefull:
          try:
            code_text= row[1]
            prompt_code=f"Remove the comments from '{code_text}' then give me 5 paraphrases changing to equivalent arithmetic or logical operators and moving the positions of the variables without altering the logic of the loop. Give me the answer in JSON format (just json, nothing else) with each paraphrase in a sequential dictionary like this: 'codes': [{{'ex1':, 'ex2':, 'ex3':, 'ex4':,'ex5':}}']"
            print("Execute "+prompt_code)
            result_code = query_perplexity(prompt_code)
            print("result code\n\n",result_code)
            result_json=json.loads(result_code)
            print("Succefully")
            succefull=True
          except:
            print("Error")
            time.sleep(2)

        for rp in results_problem:
          for i,rc in enumerate( result_json["codes"]):
            problem_description.append(row[0])
            python_code.append(row[1])
            source.append(row[2])
            description_error.append(row[3])
            error_label.append(row[4])

            print("Problem: ",rp)
            print("Code: ",rc[f'ex{i+1}'])

            problems_aug.append(rp)
            descriptions_aug.append(rc[f'ex{i+1}'])



        print("Please wait")
        time.sleep(3)

    df_result=pd.DataFrame()
    df_result['Problem Description']=problem_description
    df_result['Python Code']=python_code
    df_result['Source']=source
    df_result['Description Error']=description_error
    df_result['Error Label']=error_label
    df_result['Problem Augmented']=problems_aug
    df_result['Description Augmented']=descriptions_aug




    return df_result


# Load Dataset

In [1]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ExternData.csv'
train_full = pd.read_csv(archivo_3)

NameError: name 'pd' is not defined

In [ ]:
train_full

,Problem Description,Python Code,Source,Description Error,Error Label
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2


# Test problem

In [ ]:
text="Check if number is perfect"
prompt = f"Give me 5 paraphrases of '{text}' in comma-separated format"

In [ ]:
prompt

"Give me 5 paraphrases of 'Check if number is perfect' in comma-separated format"

In [ ]:
result = query_perplexity(prompt)


In [ ]:
result

'Determine whether a number is perfect, Test if a number qualifies as perfect, Verify if a number fits the criteria for perfect numbers, Assess if a number is a perfect number, Evaluate whether a number is perfect'

In [ ]:
import re

re.split(r',\s+\n',result)

['Determine whether a number is perfect, Test if a number qualifies as perfect, Verify if a number fits the criteria for perfect numbers, Assess if a number is a perfect number, Evaluate whether a number is perfect']

# Test code

In [ ]:
code=train_full['Python Code'][0]
prompt = f"Remove the comments from '{code}' then give me 5 paraphrases changing to equivalent arithmetic or logical operators and moving the positions of the variables without altering the logic of the cycle. Give me the answer only JSON format with each paraphrase in a sequential dictionary like this: 'codes': [{{'ex1':, 'ex2':, 'ex3':, 'ex4':,'ex5':}}']"

In [ ]:
prompt

"Remove the comments from 'def count_evens_buggy(n):\\n    i = 0\\n    count = 0\\n    while i <= n:\\n        if i % 2 == 0:\\n            count += 1\\n        # missing i += 1\\n    return count' then give me 5 paraphrases changing to equivalent arithmetic or logical operators and moving the positions of the variables without altering the logic of the cycle. Give me the answer only JSON format with each paraphrase in a sequential dictionary like this: 'codes': [{'ex1':, 'ex2':, 'ex3':, 'ex4':,'ex5':}']"

In [ ]:
result = query_perplexity(prompt)


In [ ]:
result

'{\n  "codes": [\n    {\n      "ex1": "def count_evens_buggy(n):\\n    i = 0\\n    count = 0\\n    while not (i > n):\\n        if not (i & 1):\\n            count += 1\\n        # i += 1 is missing\\n    return count"\n    },\n    {\n      "ex2": "def count_evens_buggy(n):\\n    count = 0\\n    i = 0\\n    while i <= n:\\n        if (i % 2 == 0):\\n            count = count + 1\\n        # missing i = i + 1\\n    return count"\n    },\n    {\n      "ex3": "def count_evens_buggy(n):\\n    count = 0\\n    idx = 0\\n    while idx <= n:\\n        if (idx % 2) == 0:\\n            count += 1\\n        # idx += 1 missing\\n    return count"\n    },\n    {\n      "ex4": "def count_evens_buggy(n):\\n    i = 0\\n    total = 0\\n    while i <= n:\\n        if (i & 1) == 0:\\n            total += 1\\n        # i += 1 not present\\n    return total"\n    },\n    {\n      "ex5": "def count_evens_buggy(n):\\n    counter = 0\\n    i = 0\\n    while i <= n:\\n        if (not i % 2):\\n            coun

In [ ]:
import json
result_json=json.loads(result)

In [ ]:
result_json

{'codes': [{'ex1': 'def count_evens_buggy(n):\n    i = 0\n    count = 0\n    while not (i > n):\n        if not (i & 1):\n            count += 1\n        # i += 1 is missing\n    return count'},
  {'ex2': 'def count_evens_buggy(n):\n    count = 0\n    i = 0\n    while i <= n:\n        if (i % 2 == 0):\n            count = count + 1\n        # missing i = i + 1\n    return count'},
  {'ex3': 'def count_evens_buggy(n):\n    count = 0\n    idx = 0\n    while idx <= n:\n        if (idx % 2) == 0:\n            count += 1\n        # idx += 1 missing\n    return count'},
  {'ex4': 'def count_evens_buggy(n):\n    i = 0\n    total = 0\n    while i <= n:\n        if (i & 1) == 0:\n            total += 1\n        # i += 1 not present\n    return total'},
  {'ex5': 'def count_evens_buggy(n):\n    counter = 0\n    i = 0\n    while i <= n:\n        if (not i % 2):\n            counter = counter + 1\n        # i += 1 omitted\n    return counter'}]}

In [ ]:
print(result_json['codes'][0]['ex1'])

def count_evens_buggy(n):
    i = 0
    count = 0
    while not (i > n):
        if not (i & 1):
            count += 1
        # i += 1 is missing
    return count


# Test everything

In [ ]:
train_full

,Problem Description,Python Code,Source,Description Error,Error Label
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2


In [ ]:
for i, row in train_full.iterrows():
  print(i,row[0])


0 Count how many even numbers are up to n
1 Sum numbers from 1 to n
2 Prompt user for numbers until 0 is entered
3 Find first multiple of 7 greater than n
4 Calculate factorial of n
5 Count digits in a positive integer n
6 Sum all even numbers up to n
7 Find a prime number using break
8 Print cubes of numbers from 1 to n
9 Prompt for password until correct
10 Print numbers from 1 to n, but condition excludes n
11 Divide a number by 2 repeatedly until it's less than 1
12 Sum all digits of n
13 Count how many times 5 can be subtracted from n
14 Calculate power using while loop
15 Check if number is perfect
16 Convert decimal to binary
17 Print items in list until negative is found
18 Count multiples of 3 up to n
19 Print odd numbers up to n
20 Sum the first n natural numbers
21 Sum odd numbers up to n
22 Count primes up to n
23 Read and sum numbers until a negative is entered
24 Find LCM of two numbers
25 Print Fibonacci sequence up to n terms
26 Count multiples of 5 up to n
27 Find GCD 

/tmp/ipython-input-1784881177.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(i,row[0])


In [ ]:
train_full.iloc[30:35,0]

,Problem Description
30,"Print the factorial of n, but n is not initial..."
31,Reverse a list but forget to initialize the in...
32,"Print numbers from a to b, missing both variab..."
33,"Sum even numbers in a list, with an off-by-one..."
34,"Count numbers divisible by 3, fails to initial..."


In [ ]:
train_full.iloc[30,0]='Print the factorial of n'
train_full.iloc[31,0]='Reverse a list'
train_full.iloc[32,0]='Print numbers from a to b'
train_full.iloc[33,0]='Sum even numbers in a list'
train_full.iloc[34,0]='Count numbers divisible by 3'

In [ ]:
train_full.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ExAuCorrection.csv',
                  index=False)

In [ ]:
train_full=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ExAuCorrection.csv')

In [ ]:
train_full

,Problem Description,Python Code,Source,Description Error,Error Label
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2


In [ ]:
df_augmented = augment_dataframe(train_full)

row  1
Execute Give me 5 paraphrases of 'Count how many even numbers are up to n' in comma-separated format. like this paraphrase1,paraphrase2,paraphrase3, paraphrase4,paraphrase5


/tmp/ipython-input-2253495512.py:43: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  problem_text = row[0]


result problem

 ['Determine the number of even numbers less than or equal to n', 'Find out how many even numbers exist up to n', 'Calculate the total even numbers from 1 to n', 'Count the even integers up to n', 'Figure out how many even numbers are there up to n'] 
tam: 5
Succefully


/tmp/ipython-input-2253495512.py:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  code_text= row[1]


Execute Remove the comments from 'def count_evens_buggy(n):\n    i = 0\n    count = 0\n    while i <= n:\n        if i % 2 == 0:\n            count += 1\n        # missing i += 1\n    return count' then give me 5 paraphrases changing to equivalent arithmetic or logical operators and moving the positions of the variables without altering the logic of the loop. Give me the answer in JSON format (just json, nothing else) with each paraphrase in a sequential dictionary like this: 'codes': [{'ex1':, 'ex2':, 'ex3':, 'ex4':,'ex5':}']
result code

 {
  "codes": [
    {
      "ex1": "def count_evens_buggy(n):\n    count = 0\n    i = 0\n    while i <= n:\n        if (i & 1) == 0:\n            count += 1\n        i += 1\n    return count"
    },
    {
      "ex2": "def count_evens_buggy(n):\n    i = 0\n    total = 0\n    while not (i > n):\n        if not i % 2:\n            total += 1\n        i += 1\n    return total"
    },
    {
      "ex3": "def count_evens_buggy(n):\n    i = 0\n    c = 0\n 

/tmp/ipython-input-2253495512.py:78: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  problem_description.append(row[0])
/tmp/ipython-input-2253495512.py:79: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  python_code.append(row[1])
/tmp/ipython-input-2253495512.py:80: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  source.append(row[2])
/tmp/ipython-input-2253495512.py:81: FutureWarning: Series.__getitem__ treating keys as positions is de

Streaming output truncated to the last 5000 lines.
    while 1 < n:
        result = n * result
        n = n - 2
    return result
Problem:  determine n factorial
Code:  def factorial_buggy(n):
    result = 1
    while 1 < n:
        result = n * result
        n = n - 2
    return result
Problem:  find the value of n!
Code:  def factorial_buggy(n):
    result = 1
    while 1 < n:
        result = n * result
        n = n - 2
    return result
Problem:  work out the factorial for n
Code:  def factorial_buggy(n):
    result = 1
    while 1 < n:
        result = n * result
        n = n - 2
    return result
Problem:  evaluate the factorial of n
Code:  def factorial_buggy(n):
    result = 1
    while 1 < n:
        result = n * result
        n = n - 2
    return result
Please wait
row  6
Execute Give me 5 paraphrases of 'Count digits in a positive integer n' in comma-separated format. like this paraphrase1,paraphrase2,paraphrase3, paraphrase4,paraphrase5
result problem

 ['Determine th

In [ ]:
df_augmented

,Problem Description,Python Code,Source,Description Error,Error Label,Problem Augmented,Description Augmented
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n count = 0\n ...
1,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n i = 0\n tota...
2,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n i = 0\n c = ...
3,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n count = 0\n ...
4,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n idx = 0\n re...
...,...,...,...,...,...,...,...
630,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...
631,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...
632,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...
633,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...


In [ ]:
df_augmented.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ExAuCorrectionAug.csv',
                  index=False)

In [ ]:
df_augmented=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ExAuCorrectionAug.csv')

In [ ]:
df_augmented

,Problem Description,Python Code,Source,Description Error,Error Label,Problem Augmented,Description Augmented
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n count = 0\n ...
1,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n i = 0\n tota...
2,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n i = 0\n c = ...
3,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n count = 0\n ...
4,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,Determine the number of even numbers less than...,def count_evens_buggy(n):\n idx = 0\n re...
...,...,...,...,...,...,...,...
630,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...
631,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...
632,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...
633,Count numbers divisible by 3,def buggy_count_mult_3(n):\n while i < n: ...,"Simultaneous initialization, condition, and bo...","multiple errors in initialization, condition a...",6,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...
